# Weather ETL with Polars + Delta Lake
This notebook demonstrates the full hourly → daily ETL pipeline, Delta Lake storage, time travel, schema evolution, and global aggregations.

## Install Dependencies

In [ ]:
!sudo pip install polars deltalake pandas

## Generate Sample Hourly Weather CSV

In [ ]:
import polars as pl
from datetime import datetime, timedelta
import random

CITIES = [
    ("Mumbai","MH","India",19.0760,72.8777),
    ("Delhi","DL","India",28.7041,77.1025),
    ("Bengaluru","KA","India",12.9716,77.5946),
    ("Kolkata","WB","India",22.5726,88.3639),
    ("Chennai","TN","India",13.0827,80.2707),
]

rows = []
start = datetime(2025,5,15,0,0)

for city_name, region, country, lat, lon in CITIES:
    for h in range(72):
        ts = start + timedelta(hours=h)
        temp = round(20 + 10 * random.random(),2)
        humidity = round(50 + 40 * random.random(),2)
        condition = random.choice(["Sunny","Cloudy","Rain","Clear","Partly Cloudy"])
        rows.append({
            "date": ts.strftime("%Y-%m-%d"),
            "time": ts.strftime("%Y-%m-%d %H:%M"),
            "temperature": temp,
            "condition": condition,
            "humidity": humidity,
            "location_name": city_name,
            "region": region,
            "country": country,
            "latitude": lat,
            "longitude": lon,
            "local_time": ts.strftime("%Y-%m-%d %H:%M:%S")
        })

df = pl.DataFrame(rows)
df.write_csv("hourly_weather.csv")
df.head()

## ETL: Convert hourly → daily and write Delta Lake table

In [ ]:
import polars as pl
from deltalake import write_deltalake

df = pl.read_csv("hourly_weather.csv")

df = df.with_columns([
    pl.col("time").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M").alias("timestamp")
])

df = df.with_columns([
    pl.col("timestamp").dt.date().alias("day")
])

df_daily = (
    df.group_by(["location_name","day"])
    .agg([
        pl.col("temperature").mean().round(2).alias("avg_temp"),
        pl.col("humidity").mean().round(2).alias("avg_humidity"),
        pl.count().alias("hourly_count")
    ])
)

write_deltalake("/mnt/data/delta_weather_daily", df_daily.to_pandas(), partition_by=["location_name","day"], mode="overwrite")

df_daily.head()

## Append new daily rows (schema evolution demonstration)

In [ ]:
import polars as pl
from deltalake import write_deltalake

df_new = pl.DataFrame({
    "location_name": ["Mumbai"],
    "day": ["2025-05-18"],
    "avg_temp": [33.1],
    "avg_humidity": [70.2],
    "weather_summary": [{"Sunny": 20, "Cloudy": 4}],
    "hourly_count": [24]
})

write_deltalake("/mnt/data/delta_weather_daily", df_new.to_pandas(), mode="append", schema_mode="merge")

df_new

## Time Travel and Version Reads

In [ ]:
import polars as pl
from deltalake import DeltaTable

# Latest
latest = pl.read_delta("/mnt/data/delta_weather_daily")
display(latest.head())

# Read version 0
dt = DeltaTable("/mnt/data/delta_weather_daily")
v0 = pl.from_arrow(dt.to_pyarrow_table(version=0))
display(v0.head())

dt.version()

## Global City-Level Summary

In [ ]:
import polars as pl

df = pl.read_delta("/mnt/data/delta_weather_daily")

df_global = (
    df.group_by("location_name")
      .agg([
          pl.col("avg_temp").mean().round(2).alias("global_avg_temp"),
          pl.col("avg_humidity").mean().round(2).alias("global_avg_humidity"),
          pl.count().alias("days_observed")
      ])
)

df_global